In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import os
import pickle

# Read meta info

In [2]:
PATH_TO_META = '/home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata'
files = os.listdir(PATH_TO_META)

In [3]:
perturbagen_set = set()
for file in files:
    meta = pd.read_parquet(f'{PATH_TO_META}/{file}/metadata.parquet')
    meta = meta[meta['split'] == 'train'].reset_index()
    perturbagen_set.update(set(np.hstack(meta['perturbations'].values)))

In [112]:
lpm_train = {}
for file in files:
    pert_per_context = set()
    meta = pd.read_parquet(f'{PATH_TO_META}/{file}/metadata.parquet')
    meta = meta[meta['split'] == 'train'].reset_index()
    context = meta['context'].iloc[0]
    pert_per_context.update(set(np.hstack(meta['perturbations'].values)))
    lpm_train[context] = list(pert_per_context)

# OP3

In [30]:
#PubChemCIDs were obtained with the Chem-PerturBridge pipeline with the PubChemPy package
df_emb_op3 = pd.read_csv('../../files/df_pubchem_op3.csv')
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

In [31]:
ratio = 0.25

In [32]:
op3_train_subsample = ad.read_h5ad('../../data/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
op3_test_subsample = ad.read_h5ad('../../data/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')
df = pd.read_csv('../../data/benchmark/resources/datasets/neurips-2023-data/id_map.csv')

In [33]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [34]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [35]:
compounds = np.array(op3_subsample.obs['sm_name'].unique())

In [36]:
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)

In [37]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [38]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
#op3_train_subsample_.write_h5ad('../data/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad', compression='gzip')

In [39]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
#op3_test_subsample_.write_h5ad('../data/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad', compression='gzip')

In [40]:
#op3_test_subsample_.obs[['sm_name', 'cell_type']].reset_index(drop=True).reset_index().rename(columns={'index': 'id'}).to_csv('../data/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv', index=False)

In [41]:
assert len(set(op3_test_subsample_.obs['sm_name']).intersection(set(op3_train_subsample_.obs['sm_name']))) == 0

In [42]:
len(op3_test_subsample_.obs['sm_name'].unique()) + len(op3_train_subsample_.obs['sm_name'].unique())

140

In [50]:
op3_train_subsample_pubchem = op3_train_subsample_.obs.merge(df_emb_op3, how='left', left_on='sm_name', right_on='perturbagen')

In [114]:
len(set(op3_train_subsample_pubchem['pubchem_cid'].unique()).intersection(perturbagen_set))

101

In [97]:
len(op3_train_subsample_pubchem['pubchem_cid'].unique())

105

#